# Data Preprocessing

In [73]:
import pandas as pd  #for data manipulation and analysis.
import numpy as np  #for numerical operations
from sklearn.preprocessing import StandardScaler #standardize numerical features.

**Reading and showing data information**

In [74]:
df = pd.read_csv("/ai_workforce_displacement_dirty.csv")
df.head()  #Display the first 5 rows
print("Dataset Shape:", df.shape)  #number of rows and columns
print("\nColumn Names:")
print(df.columns.tolist())   # Display column names
print("\nData Types:")
print(df.dtypes)         # Display data types of all columns
print("\nDataset Information:")
df.info()   # Get general information about the dataset

Dataset Shape: (21112, 23)

Column Names:
['record_id', 'country', 'iso3_code', 'region', 'income_group', 'year', 'quarter', 'quarter_label', 'industry_sector', 'sector_automation_risk_score', 'gdp_per_capita_usd', 'ai_adoption_index', 'pct_sector_workforce_displaced', 'pct_sector_workforce_new_roles_created', 'net_workforce_change_pct', 'ai_cited_layoff_announcements', 'ai_skill_wage_premium_pct', 'pct_workforce_female', 'pct_displaced_roles_female', 'reskilling_programs_count', 'govt_ai_policy_score_1_to_10', 'ai_tool_adoption_pct', 'data_source_notes']

Data Types:
record_id                                   int64
country                                    object
iso3_code                                  object
region                                     object
income_group                               object
year                                        int64
quarter                                    object
quarter_label                              object
industry_sector          

###Fixing missing values and duplicates


In [75]:
missing_values = df.isnull().sum()
print("Missing Values:")
print(missing_values)   # Check the number of missing values in each column

duplicate_rows = df.duplicated().sum() # Check for completely duplicated rows
print("Number of duplicated rows:", duplicate_rows)
df = df.drop_duplicates() #remove dup rows
print("Shape after removing duplicate rows:", df.shape)

Missing Values:
record_id                                   0
country                                     0
iso3_code                                   0
region                                      0
income_group                                0
year                                        0
quarter                                     0
quarter_label                               0
industry_sector                             0
sector_automation_risk_score                0
gdp_per_capita_usd                        633
ai_adoption_index                         633
pct_sector_workforce_displaced              0
pct_sector_workforce_new_roles_created      0
net_workforce_change_pct                    0
ai_cited_layoff_announcements               0
ai_skill_wage_premium_pct                   0
pct_workforce_female                      633
pct_displaced_roles_female                  0
reskilling_programs_count                 633
govt_ai_policy_score_1_to_10                0
ai_tool_adoption_p

###Check whether record_id contains duplicated values

In [76]:
duplicate_ids = df["record_id"].duplicated().sum()
print("Duplicated Record IDs:", duplicate_ids)
# Check the number of unique record IDs
print("Unique Record IDs:", df["record_id"].nunique())
df = df.drop_duplicates(subset="record_id", keep="first")  # Keep only the first occurrence of each Record ID
print("Shape after removing duplicate rows:", df.shape)

Duplicated Record IDs: 162
Unique Record IDs: 20800
Shape after removing duplicate rows: (20800, 23)


###Remove leading and trailing spaces from categorical columns

In [77]:
# Remove leading and trailing spaces from categorical columns
categorical_columns = df.select_dtypes(include="object").columns
print("Categorical Columns:")
print(categorical_columns)
for col in categorical_columns:
    df[col] = df[col].astype(str).str.strip()  # Remove leading and trailing spaces from categorical columns
for col in categorical_columns:
    has_spaces = df[col].astype(str).str.match(r'^\s|\s$').sum()
    print(f"{col}: {has_spaces} values with extra spaces")   # Validate that no extra leading or trailing spaces remain

Categorical Columns:
Index(['country', 'iso3_code', 'region', 'income_group', 'quarter',
       'quarter_label', 'industry_sector', 'gdp_per_capita_usd',
       'pct_sector_workforce_displaced', 'data_source_notes'],
      dtype='object')
country: 0 values with extra spaces
iso3_code: 0 values with extra spaces
region: 0 values with extra spaces
income_group: 0 values with extra spaces
quarter: 0 values with extra spaces
quarter_label: 0 values with extra spaces
industry_sector: 0 values with extra spaces
gdp_per_capita_usd: 0 values with extra spaces
pct_sector_workforce_displaced: 0 values with extra spaces
data_source_notes: 0 values with extra spaces


###Fixing typos and standardizing country names

In [78]:
#Fixing typos and standardizing country names
typo_fix = {    #counry name cleanup
    "Untied States": "United States",
    "Chna": "China",
    "Gemany": "Germany",
    "Brasil": "Brazil",
}
df["country"] = df["country"].replace(typo_fix)
df["country"] = df["country"].str.title()  #capitalize the first letter of each word in the country names
title_fix = {"United States Of America": "United States"}  # example guard, extend as needed
df["country"] = df["country"].replace(title_fix)

###Normalizing the quarter and quarter_label columns

In [79]:
print(df["quarter"].unique())
print(df["quarter"].dtype)
print(df["quarter_label"].unique())
df["quarter"] = (
    df["quarter"]
    .astype(str)
    .str.upper()
    .str.replace("Q", "", regex=False)   #Drop the Q and keep the number only
    .astype(int)
)
df["quarter_label"] = (
    df["year"].astype(int).astype(str) + "-Q" + df["quarter"].astype(str)
)  #I standardized the quarter_label format to YYYY-QN using the year and quarter columns
print(df["quarter"].unique())
print(df["quarter"].dtype)
print(df["quarter_label"].unique())

['2' '4' '3' '1' 'Q1' 'Q2' 'Q3' 'Q4']
object
['2026-Q2' '2020-Q4' '2020-Q3' '2024-Q1' '2024-Q4' '2021-Q4' '2022-Q1'
 '2021-Q3' '2023-Q2' '2022/Q3' '2021-Q2' '2020/Q1' '2022-Q2' '2026/Q2'
 '2025-Q2' '2024-Q2' '2022-Q3' '2022-Q4' '2026-Q1' '2020-Q1' '2024/Q4'
 '2025-Q4' '2023-Q3' '2025-Q3' 'Q1 2020' '2023-Q4' '2023-Q1' '2025-Q1'
 '2023/Q4' '2020-Q2' '2021-Q1' '2025/Q1' '2020/Q2' '2024-Q3' '2021/Q3'
 'Q3 2021' 'Q3 2022' 'Q2 2024' '2023/Q1' '2023/Q3' '2021/Q2' '2026/Q1'
 'Q1 2021' 'Q4 2022' '2022/Q1' '2025/Q3' 'Q1 2022' '2025/Q4' 'Q4 2023'
 '2022/Q4' 'Q3 2025' 'Q4 2020' 'Q1 2025' 'Q1 2023' 'Q1 2024' '2020/Q4'
 'Q3 2024' 'Q1 2026' 'Q2 2020' 'Q3 2020' '2024/Q1' '2022/Q2' 'Q4 2024'
 '2023/Q2' 'Q4 2021' 'Q2 2025' 'Q2 2023' '2025/Q2' 'Q2 2026' 'Q2 2021'
 '2024/Q3' '2021/Q1' 'Q3 2023' '2024/Q2' '2021/Q4' 'Q4 2025' '2020/Q3'
 'Q2 2022']
[2 4 3 1]
int64
['2026-Q2' '2020-Q4' '2020-Q3' '2024-Q1' '2024-Q4' '2021-Q4' '2022-Q1'
 '2021-Q3' '2023-Q2' '2022-Q3' '2021-Q2' '2020-Q1' '2022-Q2' '2025-Q2'
 '20

###Normalizing the region_map column

In [80]:
region_map = {
    "N. America": "North America",
    "n. america": "North America",
    "north america": "North America",
    "North America ": "North America",
    "EU": "Europe",
    "europe": "Europe",
}
df["region"] = df["region"].replace(region_map).str.strip()

###Normalising Values in the gdp and pct columns

###For the gdp_per_capita_usd column some values begin with '\$' and some don't. Normalize by removing "\$" and converting to float

In [81]:
df["gdp_per_capita_usd"] = (
    df["gdp_per_capita_usd"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

###for the pct_sector_workforce_displaced column some values are in percentage format while others are in decimal format. Normalize by converting all values to decimal format.

In [82]:
df["pct_sector_workforce_displaced"] = (
    df["pct_sector_workforce_displaced"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .astype(float) / 100
)

###Check the sector_automation_risk_score column for invalid values outside the allowed range [0, 1]

In [83]:
bad_risk = ~df["sector_automation_risk_score"].between(0, 1)
print("Invalid values found:", bad_risk.sum())
# Replace invalid values with NaN
df.loc[bad_risk, "sector_automation_risk_score"] = np.nan
print(df["sector_automation_risk_score"].describe())
print("Missing values:", df["sector_automation_risk_score"].isna().sum())

Invalid values found: 14
count    20786.000000
mean         0.537897
std          0.190338
min          0.151000
25%          0.379000
50%          0.580000
75%          0.698000
max          0.855000
Name: sector_automation_risk_score, dtype: float64
Missing values: 14


###Checking if there's any negative value for the layoff announcments

In [84]:
bad_layoffs = df["ai_cited_layoff_announcements"] < 0
print(f"Invalid negative ai_cited_layoff_announcements found: {bad_layoffs.sum()}")
df.loc[bad_layoffs, "ai_cited_layoff_announcements"] = np.nan   #I identified negative layoff announcement counts as invalid and replaced them with NaN

Invalid negative ai_cited_layoff_announcements found: 10


###data_source_notes column is a repeated string but some values are missing, so fill them upfill Nan with "Research-calibrated synthetic data. Grounded in: WEF Future of Jobs 2025; Goldman Sachs GenAI Labour Report 2025; McKinsey State of AI 2025; OECD Employment Outlook 2025; BLS O*NET Automation Scores; Layoffs.fyi 2025; PwC AI Jobs Barometer 2025; IMF WEO 2025."

In [85]:
missing = df.isna().sum()
if missing["data_source_notes"] > 0:
    canonical_note = (
        "Research-calibrated synthetic data. Grounded in: WEF Future of Jobs 2025; "
        "Goldman Sachs GenAI Labour Report 2025; McKinsey State of AI 2025; "
        "OECD Employment Outlook 2025; BLS O*NET Automation Scores; Layoffs.fyi 2025; "
        "PwC AI Jobs Barometer 2025; IMF WEO 2025."
    )
    df["data_source_notes"].fillna(canonical_note, inplace=True)

###Check for remaining missing values and handling them
Handling missing counts using the general median will cause the data to be skewed when ranked by country, industry and year.
The most optimal solution is to use the median or mean of the parameter that affects the numbers the most.

In [86]:
df = df.sort_values(['country', 'year', 'quarter']).reset_index(drop=True)
# --- Group 1: slow-moving, time-based columns -> interpolate within each country ---
time_based_cols = ['gdp_per_capita_usd', 'ai_adoption_index', 'reskilling_programs_count']
for col in time_based_cols:
    df[col] = df.groupby('country')[col].transform(
        lambda s: s.interpolate(method='linear', limit_direction='both')
    )
    # Fallback: if a country had NO data at all for a column, interpolation
    # can't help — fall back to that column's global median so nothing is left NaN
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())
# --- Group 2: sector-driven traits -> fill with the sector's median ---
sector_based_cols = ['sector_automation_risk_score', 'pct_workforce_female']
for col in sector_based_cols:
    df[col] = df[col].fillna(df.groupby('industry_sector')[col].transform('median'))
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())
# --- Group 3: sparse counts -> median within sector + year ---
df['ai_cited_layoff_announcements'] = df['ai_cited_layoff_announcements'].fillna(
    df.groupby(['industry_sector', 'year'])['ai_cited_layoff_announcements'].transform('median')
)
if df['ai_cited_layoff_announcements'].isna().any():
    df['ai_cited_layoff_announcements'] = df['ai_cited_layoff_announcements'].fillna(
        df['ai_cited_layoff_announcements'].median())
missing = df.isna().sum()
print("\nRemaining missing values by column:")
print(missing[missing > 0])


Remaining missing values by column:
Series([], dtype: int64)


###Save clean dataset

In [87]:
df.to_csv("ai_workforce_displacement_clean.csv", index=False)
print("\nCleaned data shape:", df.shape)


Cleaned data shape: (20800, 23)


In [88]:
from google.colab import files
files.download("ai_workforce_displacement_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>